# 09m: CLIP RN50x4 Vehicle ReID Training on CityFlowV2

Fine-tune the **OpenAI CLIP RN50x4** visual encoder on **CityFlowV2** with a ReID head built as **Linear(640->768) -> BNNeck -> classifier**, using the proven CityFlowV2 crop extraction pipeline, mixed precision, EMA, CE with label smoothing, triplet loss, and delayed center loss.

## Experiment goals
1. Train an architecturally diverse CNN-based CLIP model to complement the ViT-based 09l / 10c ensemble.
2. Keep the notebook fully self-contained on Kaggle, including CityFlowV2 download, crop extraction, training, evaluation, and export.
3. Measure **mAP / Rank-1** on a CityFlowV2 identity split with original + flipped inference.
4. Export the best EMA checkpoint for stage 2 integration.

## Training recipe
1. Load `open_clip.create_model("RN50x4", pretrained="openai")` and keep only the visual encoder.
2. Use native **288x288** input resolution for RN50x4 attention pooling.
3. Train with **AdamW**, cosine LR with warmup, **EMA**, **CE+LS(0.05)**, **Triplet(m=0.3)**, and delayed **CenterLoss**.
4. Build train / query / gallery splits from CityFlowV2 GT crops using the same protocol as 09l.
5. Export the best EMA checkpoint and training metadata to `/kaggle/working/exported_models/`.

## Expected outputs
- `working/09m_clip_rn50x4/output/clip_rn50x4_cityflowv2_best_ema.pth`
- `working/09m_clip_rn50x4/output/clip_rn50x4_cityflowv2_last_ema.pth`
- `working/09m_clip_rn50x4/output/training_curves.png`
- `working/exported_models/clip_rn50x4_cityflowv2.pth`
- `working/exported_models/clip_rn50x4_cityflowv2_metadata.json`

In [ ]:
%pip install -q open_clip_torch timm matplotlib pandas loguru gdown

## 1. Setup

In [ ]:
import copy
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import open_clip
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.optim import SGD
from torch.utils.data import DataLoader, Dataset, Sampler

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
NUM_GPUS = max(torch.cuda.device_count(), 1)
print(f"GPUs available: {NUM_GPUS}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"  GPU {index}: {torch.cuda.get_device_name(index)} ({props.total_memory / 1024**3:.1f} GB)")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

INPUT_SIZE = (288, 288)
BACKBONE_OUT_DIM = 640
EMBED_DIM = 768
BATCH_SIZE = 64
P = 16
K = 4
EPOCHS = 200
WARMUP_EPOCHS = 10
BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 5e-4
EMA_DECAY = 0.9999
CENTER_WEIGHT = 5e-4
CENTER_START_EPOCH = 15
EVAL_EVERY = 20
NUM_WORKERS = 4

OUTPUT_DIR = Path("/kaggle/working/09m_clip_rn50x4/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = Path("/kaggle/working/exported_models")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nDevice: {DEVICE}")
print(f"Input size: {INPUT_SIZE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Export dir: {EXPORT_DIR}")

## 1.5 Download CityFlowV2 from Google Drive

CityFlowV2 is not available as a public Kaggle dataset here, so the notebook downloads it from Google Drive at runtime and flattens the camera directories exactly as in 09l.

In [ ]:
# -- Download CityFlowV2 from Google Drive --
# All large files go to /tmp/ to avoid filling /kaggle/working/ (~20GB limit).
import re as _re

CITYFLOW_DIR = Path("/tmp/cityflowv2")
GDRIVE_ID = "13wNJpS_Oaoe-7y5Dzexg_Ol7bKu1OWuC"
ARCHIVE_NAME = "AIC22_Track1_MTMC_Tracking.zip"
ALLOWED_SPLITS = {"train", "validation"}

already_found = False
for check_dir in [CITYFLOW_DIR, Path("/kaggle/input/cityflowv2"), Path("/kaggle/input/aic22-track1-mtmc-tracking")]:
    if check_dir.exists() and any(list(check_dir.rglob("vdo.avi"))[:1]):
        print(f"CityFlowV2 already available at {check_dir}")
        CITYFLOW_DIR = check_dir
        already_found = True
        break

if not already_found:
    CITYFLOW_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = Path(f"/tmp/{ARCHIVE_NAME}")

    if not archive_path.exists():
        print(f"Downloading CityFlowV2 from Google Drive (id={GDRIVE_ID})...")
        print("This may take 10-20 minutes for the full dataset (~20GB).")
        import gdown

        gdown.download(
            f"https://drive.google.com/uc?id={GDRIVE_ID}",
            str(archive_path),
            quiet=False,
        )
    else:
        print(f"Archive already downloaded: {archive_path}")

    staging = Path("/tmp/_aic22_staging")
    staging.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_path} to {staging}...")
    with zipfile.ZipFile(str(archive_path), "r") as zf:
        zf.extractall(str(staging))

    if archive_path.exists():
        archive_path.unlink()
        print("Deleted archive to reclaim space.")

    print("\nExtracted contents of staging dir:")
    for item in sorted(staging.iterdir()):
        kind = "dir" if item.is_dir() else "file"
        print(f"  {item.name} ({kind})")

    moved = 0
    skipped_splits = set()
    for vdo_path in sorted(staging.rglob("vdo.avi")):
        cam_dir = vdo_path.parent
        scene_dir = cam_dir.parent
        split_dir = scene_dir.parent
        cam_name = cam_dir.name
        scene_name = scene_dir.name
        split_name = split_dir.name
        if not cam_name.startswith("c") or not scene_name.startswith("S"):
            print(f"  SKIP unusual path: {vdo_path}")
            continue
        if split_name not in ALLOWED_SPLITS:
            skipped_splits.add(split_name)
            print(f"  SKIP {scene_name}_{cam_name}: {split_name} split (no GT)")
            continue
        flat_name = f"{scene_name}_{cam_name}"
        target = CITYFLOW_DIR / flat_name
        if not target.exists():
            shutil.move(str(cam_dir), str(target))
            moved += 1
            if moved <= 15:
                print(f"  {flat_name} -> {target}")
            elif moved == 16:
                print("  ... (showing first 15 only)")
    print(f"Total cameras moved: {moved}")
    if skipped_splits:
        print(f"Skipped splits: {sorted(skipped_splits)} (no GT annotations)")

    if moved == 0:
        print("Trying fallback: zip may have extracted directly to /tmp/...")
        for split_name in sorted(ALLOWED_SPLITS):
            split_dir = Path(f"/tmp/{split_name}")
            if not split_dir.exists():
                continue
            for vdo_path in sorted(split_dir.rglob("vdo.avi")):
                cam_dir = vdo_path.parent
                scene_dir = cam_dir.parent
                cam_name = cam_dir.name
                scene_name = scene_dir.name
                if not cam_name.startswith("c") or not scene_name.startswith("S"):
                    continue
                flat_name = f"{scene_name}_{cam_name}"
                target = CITYFLOW_DIR / flat_name
                if not target.exists():
                    shutil.move(str(cam_dir), str(target))
                    moved += 1
                    if moved <= 15:
                        print(f"  {flat_name} -> {target}")
        print(f"Fallback cameras moved: {moved}")

    if staging.exists():
        shutil.rmtree(str(staging), ignore_errors=True)
    for path in [Path("/tmp/train"), Path("/tmp/test"), Path("/tmp/validation")]:
        if path.exists():
            shutil.rmtree(str(path), ignore_errors=True)
    for path in [Path("/tmp/ReadMe.txt"), Path("/tmp/list_cam.txt")]:
        if path.exists():
            path.unlink()
    for path in Path("/tmp").glob("Dataset License*"):
        path.unlink(missing_ok=True)
    for path in [Path("/tmp/cam_framenum"), Path("/tmp/cam_loc"), Path("/tmp/cam_timestamp"), Path("/tmp/eval")]:
        if path.exists():
            shutil.rmtree(str(path), ignore_errors=True)

_cam_re = _re.compile(r"^S\d{2}_c\d{3}$")
found_cams = sorted([d.name for d in CITYFLOW_DIR.iterdir() if d.is_dir() and _cam_re.match(d.name)]) if CITYFLOW_DIR.exists() else []
print(f"\nCityFlowV2 at {CITYFLOW_DIR}")
print(f"Camera dirs found: {len(found_cams)} (46 unique physical cameras)")
print(f"  {found_cams}")
if not found_cams:
    print("\nERROR: No cameras found! Listing /tmp/ for debugging:")
    for path in sorted(Path("/tmp").iterdir()):
        if not path.name.startswith("uv-") and not path.name.startswith("tmp"):
            kind = "dir" if path.is_dir() else "file"
            print(f"  {path.name} ({kind})")

## 2. CityFlowV2 Dataset - Crop Extraction

In [ ]:
# -- Locate CityFlowV2 data and extract vehicle crops from GT + video --
TARGET_CAMS = set()
MAX_CROPS_PER_ID_CAM = 20
MIN_AREA = 2000
MIN_BBOX_SIDE = 30
TRAIN_RATIO = 0.7
MIN_CAMS_FOR_EVAL = 2

possible_roots = [
    CITYFLOW_DIR,
    Path("/tmp/cityflowv2"),
    Path("/kaggle/input/cityflowv2"),
    Path("/kaggle/input/aic22-track1-mtmc-tracking"),
    Path("/tmp/data/raw/cityflowv2"),
    Path("data/raw/cityflowv2"),
]

DATA_ROOT = None
for root in possible_roots:
    if root.exists():
        cam_dirs = [d for d in root.iterdir() if d.is_dir() and "_c0" in d.name]
        if cam_dirs:
            DATA_ROOT = root
            break

if DATA_ROOT is None:
    for base in [Path("/tmp"), Path("/kaggle/input")]:
        if not base.exists():
            continue
        vdos = list(base.rglob("vdo.avi"))[:1]
        if vdos:
            cam = vdos[0].parent
            DATA_ROOT = cam.parent.parent if cam.parent.name.startswith("S0") else cam.parent
            break

if DATA_ROOT is None:
    raise RuntimeError(
        "CityFlowV2 not found. Either:\n"
        "  1. Attach a CityFlowV2 dataset on Kaggle\n"
        "  2. Check the download cell output for errors\n"
        "  3. Re-run the download cell"
    )

print(f"CityFlowV2 data root: {DATA_ROOT}")

def find_gt(cam_dir):
    direct = cam_dir / "gt.txt"
    if direct.exists():
        return direct
    nested = cam_dir / "gt" / "gt.txt"
    if nested.exists():
        return nested
    matches = list(cam_dir.rglob("gt.txt"))
    return matches[0] if matches else None

def find_video(cam_dir):
    for ext in ["avi", "mp4"]:
        for name in ["vdo", "video"]:
            path = cam_dir / f"{name}.{ext}"
            if path.exists():
                return path
    vids = list(cam_dir.glob("*.avi")) + list(cam_dir.glob("*.mp4"))
    return vids[0] if vids else None

CAMERAS = []
cam_gt_paths = {}
cam_vid_paths = {}
for cam_dir in sorted(DATA_ROOT.iterdir()):
    if not cam_dir.is_dir():
        continue
    cam_name = cam_dir.name
    if TARGET_CAMS and cam_name not in TARGET_CAMS:
        continue
    gt = find_gt(cam_dir)
    vid = find_video(cam_dir)
    if gt and vid:
        CAMERAS.append(cam_name)
        cam_gt_paths[cam_name] = gt
        cam_vid_paths[cam_name] = vid
        print(f"  OK {cam_name}: GT={gt.name} Video={vid.name}")
    else:
        missing = []
        if not gt:
            missing.append("GT")
        if not vid:
            missing.append("video")
        print(f"  SKIP {cam_name}: missing {', '.join(missing)}")

if not CAMERAS:
    raise RuntimeError("No cameras with both GT and video found!")

print(f"\nUsing {len(CAMERAS)}/{len(found_cams)} cameras: {CAMERAS}")

CROP_DIR = Path("/tmp/cityflowv2_crops")
CROP_DIR.mkdir(parents=True, exist_ok=True)

def load_gt(gt_path):
    rows = []
    with open(gt_path, "r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame, tid = int(parts[0]), int(parts[1])
            x, y, w, h = int(parts[2]), int(parts[3]), int(parts[4]), int(parts[5])
            rows.append((frame, tid, x, y, w, h))
    return rows

def extract_crops_from_camera(cam_name, gt_file, vid_file, crop_dir, max_crops, min_area):
    gt_rows = load_gt(str(gt_file))
    if not gt_rows:
        print(f"  {cam_name}: empty GT file")
        return {}

    id_dets = defaultdict(list)
    for frame, tid, x, y, w, h in gt_rows:
        id_dets[tid].append((frame, x, y, w, h))

    frame_to_dets = defaultdict(list)
    for tid, dets in id_dets.items():
        if len(dets) <= max_crops:
            sampled = dets
        else:
            step = len(dets) / max_crops
            sampled = [dets[int(i * step)] for i in range(max_crops)]
        for frame, x, y, w, h in sampled:
            if w * h >= min_area and w >= MIN_BBOX_SIDE and h >= MIN_BBOX_SIDE:
                frame_to_dets[frame].append((tid, x, y, w, h))

    if not frame_to_dets:
        return {}

    crops = defaultdict(list)
    cap = cv2.VideoCapture(str(vid_file))
    current_frame = 0
    target_frames = sorted(frame_to_dets.keys())
    target_idx = 0

    while target_idx < len(target_frames):
        ret, img = cap.read()
        if not ret:
            break
        current_frame += 1
        if current_frame < target_frames[target_idx]:
            continue
        if current_frame > target_frames[target_idx]:
            while target_idx < len(target_frames) and target_frames[target_idx] < current_frame:
                target_idx += 1
            if target_idx >= len(target_frames) or current_frame != target_frames[target_idx]:
                continue

        height, width = img.shape[:2]
        for tid, x, y, w, h in frame_to_dets[current_frame]:
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(width, x + w), min(height, y + h)
            if x2 - x1 < MIN_BBOX_SIDE or y2 - y1 < MIN_BBOX_SIDE:
                continue
            crop = img[y1:y2, x1:x2]
            fname = f"{tid:04d}_{cam_name}_f{current_frame:06d}.jpg"
            fpath = str(crop_dir / fname)
            cv2.imwrite(fpath, crop)
            crops[tid].append(fpath)
        target_idx += 1

    cap.release()
    total_crops = sum(len(paths) for paths in crops.values())
    print(f"  {cam_name}: {total_crops} crops from {len(crops)} vehicles")
    return dict(crops)

existing_crops = list(CROP_DIR.glob("*.jpg"))
if len(existing_crops) > 100:
    print(f"Reusing {len(existing_crops)} existing crops from {CROP_DIR}")
    all_crops = defaultdict(lambda: defaultdict(list))
    for fpath in sorted(existing_crops):
        parts = fpath.stem.split("_")
        tid = int(parts[0])
        cam = parts[1] + "_" + parts[2]
        all_crops[tid][cam].append(str(fpath))
    all_crops = {tid: dict(cams) for tid, cams in all_crops.items()}
else:
    print("Extracting crops from videos...")
    all_crops = defaultdict(lambda: defaultdict(list))
    for cam in CAMERAS:
        cam_crops = extract_crops_from_camera(
            cam,
            cam_gt_paths[cam],
            cam_vid_paths[cam],
            CROP_DIR,
            MAX_CROPS_PER_ID_CAM,
            MIN_AREA,
        )
        for tid, paths in cam_crops.items():
            all_crops[tid][cam].extend(paths)
    all_crops = {tid: dict(cams) for tid, cams in all_crops.items()}

print("Cleaning up raw CityFlowV2 data to free disk space...")
if CITYFLOW_DIR.exists() and str(CITYFLOW_DIR).startswith("/tmp/"):
    shutil.rmtree(str(CITYFLOW_DIR), ignore_errors=True)
    print(f"  Removed {CITYFLOW_DIR}")

total = sum(sum(len(paths) for paths in cams.values()) for cams in all_crops.values())
n_ids = len(all_crops)
multi_cam = sum(1 for cams in all_crops.values() if len(cams) >= MIN_CAMS_FOR_EVAL)
print(f"\nTotal: {total} crops, {n_ids} vehicle IDs ({multi_cam} with >={MIN_CAMS_FOR_EVAL} cameras)")

scene_ids = defaultdict(set)
for tid, cams in all_crops.items():
    for cam_name in cams:
        scene = cam_name.split("_")[0]
        scene_ids[scene].add(tid)
cross_scene = 0
for tid, cams in all_crops.items():
    scenes = {cam_name.split("_")[0] for cam_name in cams}
    if len(scenes) > 1:
        cross_scene += 1
print("\nID sanity check:")
for scene in sorted(scene_ids):
    print(f"  {scene}: {len(scene_ids[scene])} vehicle IDs")
print(f"  Cross-scene vehicles (same ID in 2+ scenes): {cross_scene}")
if n_ids > 880:
    print(f"  WARNING: {n_ids} IDs found but CityFlowV2 has 880 annotated. Possible ID collision!")
else:
    print(f"  OK: {n_ids} IDs <= 880 official annotated identities")

In [ ]:
# -- Build train / query / gallery splits --
if not all_crops:
    raise RuntimeError(
        f"No crops extracted! CAMERAS={CAMERAS}, CROP_DIR={CROP_DIR}\n"
        "The download or extraction likely failed. Check the earlier cells."
    )

rng = np.random.RandomState(SEED)

multi_cam_ids = sorted(tid for tid, cams in all_crops.items() if len(cams) >= MIN_CAMS_FOR_EVAL)
single_cam_ids = sorted(tid for tid, cams in all_crops.items() if len(cams) < MIN_CAMS_FOR_EVAL)

if not multi_cam_ids:
    print(f"WARNING: No vehicles seen in >={MIN_CAMS_FOR_EVAL} cameras.")
    print("  Using single-cam IDs for training (no cross-camera eval possible).")
    all_ids = sorted(all_crops.keys())
    rng.shuffle(all_ids)
    n_train = max(int(len(all_ids) * TRAIN_RATIO), 1)
    multi_cam_ids = all_ids
    single_cam_ids = []

rng.shuffle(multi_cam_ids)
n_train = int(len(multi_cam_ids) * TRAIN_RATIO)
train_ids = set(multi_cam_ids[:n_train])
eval_ids = set(multi_cam_ids[n_train:])

cam_names = sorted({cam for cams in all_crops.values() for cam in cams})
cam2id = {cam_name: index for index, cam_name in enumerate(cam_names)}

train_data, query_data, gallery_data = [], [], []
train_pid_set = sorted(train_ids)
pid2label = {tid: index for index, tid in enumerate(train_pid_set)}

for tid in sorted(train_ids):
    label = pid2label[tid]
    for cam_name, paths in all_crops[tid].items():
        camid = cam2id[cam_name]
        for path in paths:
            train_data.append((path, label, camid))

eval_pid2label = {tid: index for index, tid in enumerate(sorted(eval_ids))}
for tid in sorted(eval_ids):
    pid = eval_pid2label[tid]
    for cam_name, paths in all_crops[tid].items():
        if not paths:
            continue
        camid = cam2id[cam_name]
        query_index = rng.randint(0, len(paths))
        query_data.append((paths[query_index], pid, camid))
        for index, path in enumerate(paths):
            if index != query_index:
                gallery_data.append((path, pid, camid))

distractor_pid = len(eval_ids)
for tid in sorted(single_cam_ids):
    for cam_name, paths in all_crops[tid].items():
        camid = cam2id[cam_name]
        for path in paths:
            gallery_data.append((path, distractor_pid, camid))
    distractor_pid += 1

num_classes = max(len(set(pid for _, pid, _ in train_data)), 1)
num_cameras = max(len(cam_names), 1)
CAN_EVAL = len(query_data) > 0 and len(gallery_data) > 0

print(f"Train:   {len(train_data)} images, {num_classes} IDs, {num_cameras} cameras")
print(f"Query:   {len(query_data)} images, {len(eval_ids)} IDs")
print(f"Gallery: {len(gallery_data)} images")
print(f"Cameras: {cam_names}")
print(f"Evaluation possible: {CAN_EVAL}")

## 3. Data Loading + Losses

In [ ]:
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]
H, W = INPUT_SIZE

train_tf = T.Compose([
    T.Resize((H + 16, W + 16), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((H, W)),
    T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0.0),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value="random"),
])

test_tf = T.Compose([
    T.Resize((H, W), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])

class ReIDDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        path, pid, cam = self.data[index]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, pid, cam, path

class PKSampler(Sampler):
    def __init__(self, data_source, p=16, k=4):
        self.data_source = data_source
        self.p = p
        self.k = k
        self.pid_to_indices = defaultdict(list)
        for index, (_, pid, _) in enumerate(data_source):
            self.pid_to_indices[pid].append(index)
        self.pids = list(self.pid_to_indices.keys())
        self.length = max(len(self.pids), 1) * self.k

    def __iter__(self):
        order = np.random.permutation(self.pids)
        batch = []
        for pid in order:
            indices = self.pid_to_indices[pid]
            replace = len(indices) < self.k
            choice = np.random.choice(indices, self.k, replace=replace)
            batch.extend(choice.tolist())
        return iter(batch)

    def __len__(self):
        return self.length

train_ds = ReIDDataset(train_data, train_tf)
query_ds = ReIDDataset(query_data, test_tf)
gallery_ds = ReIDDataset(gallery_data, test_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=PKSampler(train_data, p=P, k=K),
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)
query_loader = DataLoader(query_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)
gallery_loader = DataLoader(gallery_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Query batches: {len(query_loader)}")
print(f"Gallery batches: {len(gallery_loader)}")

In [ ]:
class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.05):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon

    def forward(self, inputs, targets):
        log_probs = F.log_softmax(inputs.float(), dim=1)
        with torch.no_grad():
            one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
            smooth = (1 - self.epsilon) * one_hot + self.epsilon / self.num_classes
        return (-smooth * log_probs).sum(dim=1).mean()

class TripletLossHardMining(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, feats, pids):
        feats = F.normalize(feats.float(), p=2, dim=1)
        num = feats.size(0)
        dist = torch.cdist(feats, feats, p=2)
        mask_pos = pids.unsqueeze(0).eq(pids.unsqueeze(1))
        mask_neg = ~mask_pos

        dist_pos = dist.clone()
        dist_pos[~mask_pos] = 0
        hardest_pos, _ = dist_pos.max(dim=1)

        dist_neg = dist.clone()
        dist_neg[~mask_neg] = float("inf")
        hardest_neg, _ = dist_neg.min(dim=1)

        target = torch.ones(num, device=feats.device)
        return self.ranking_loss(hardest_neg, hardest_pos, target)

class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

    def forward(self, feats, labels):
        feats = feats.float()
        centers_batch = self.centers[labels]
        diff = feats - centers_batch
        return (diff * diff).sum(dim=1).mean()

class ModelEMA:
    def __init__(self, model, decay=0.9999):
        self.ema = copy.deepcopy(model)
        self.ema.eval()
        self.decay = decay
        for param in self.ema.parameters():
            param.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            for ema_param, param in zip(self.ema.parameters(), model.parameters()):
                ema_param.data.mul_(self.decay).add_(param.data, alpha=1 - self.decay)
            for ema_buffer, buffer in zip(self.ema.buffers(), model.buffers()):
                ema_buffer.copy_(buffer)

print("Training utilities ready: CE+LS(0.05), Triplet(m=0.3), CenterLoss, EMA")

## 4. Evaluation Functions

In [ ]:
@torch.no_grad()
def extract_features(model, loader, device="cuda", flip=True, feat_dim=EMBED_DIM):
    model.eval()
    feats, pids, cams = [], [], []
    for imgs, pid, cam, _ in loader:
        imgs = imgs.to(device)
        features = model(imgs)
        if isinstance(features, (tuple, list)):
            features = features[-1]
        if flip:
            flipped = model(torch.flip(imgs, [3]))
            if isinstance(flipped, (tuple, list)):
                flipped = flipped[-1]
            features = (features + flipped) / 2
        features = F.normalize(features, p=2, dim=1)
        feats.append(features.cpu().numpy())
        pids.append(pid.numpy())
        cams.append(cam.numpy())
    if not feats:
        return np.zeros((0, feat_dim)), np.zeros(0, dtype=int), np.zeros(0, dtype=int)
    return np.concatenate(feats), np.concatenate(pids), np.concatenate(cams)

def eval_market1501(distmat, q_pids, g_pids, q_cams, g_cams, max_rank=50):
    if distmat.shape[0] == 0 or distmat.shape[1] == 0:
        return 0.0, np.zeros(max_rank)
    nq = distmat.shape[0]
    indices = np.argsort(distmat, axis=1)
    matches = (g_pids[indices] == q_pids[:, None]).astype(np.int32)
    all_cmc, all_ap = [], []
    for q_index in range(nq):
        order = indices[q_index]
        remove = (g_pids[order] == q_pids[q_index]) & (g_cams[order] == q_cams[q_index])
        raw_cmc = matches[q_index][~remove]
        if raw_cmc.sum() == 0:
            continue
        cmc = raw_cmc.cumsum()
        cmc[cmc > 1] = 1
        clipped = np.zeros(max_rank)
        clipped[: min(max_rank, len(cmc))] = cmc[:max_rank]
        all_cmc.append(clipped)
        num_rel = raw_cmc.sum()
        tmp = raw_cmc.cumsum()
        prec = tmp / (np.arange(len(tmp)) + 1.0)
        all_ap.append((prec * raw_cmc).sum() / num_rel)
    if not all_ap:
        return 0.0, np.zeros(max_rank)
    return float(np.mean(all_ap)), np.array(all_cmc).mean(axis=0)

def compute_reranking(qf, gf, k1=20, k2=6, lam=0.3):
    if qf.shape[0] == 0 or gf.shape[0] == 0:
        return np.zeros((qf.shape[0], gf.shape[0]))
    all_f = np.concatenate([qf, gf], axis=0)
    num_all, num_query = all_f.shape[0], qf.shape[0]
    original_dist = np.clip(2.0 - 2.0 * (all_f @ all_f.T), 0, None)
    initial_rank = np.argsort(original_dist, axis=1)
    v = np.zeros((num_all, num_all), dtype=np.float32)
    for index in range(num_all):
        forward = initial_rank[index, : k1 + 1]
        reciprocal = [candidate for candidate in forward if index in initial_rank[candidate, : k1 + 1]]
        reciprocal = np.array(reciprocal)
        expanded = reciprocal.copy()
        for candidate in reciprocal:
            candidate_forward = initial_rank[candidate, : int(round(k1 / 2)) + 1]
            candidate_reciprocal = [other for other in candidate_forward if candidate in initial_rank[other, : int(round(k1 / 2)) + 1]]
            if len(candidate_reciprocal) > 2 / 3 * len(candidate_forward):
                expanded = np.union1d(expanded, candidate_reciprocal)
        weight = np.exp(-original_dist[index, expanded])
        v[index, expanded] = weight / (weight.sum() + 1e-12)
    if k2 > 0:
        v_qe = np.zeros_like(v)
        for index in range(num_all):
            v_qe[index] = v[initial_rank[index, : k2 + 1]].mean(axis=0)
        v = v_qe
    jaccard = np.zeros((num_query, num_all - num_query), dtype=np.float32)
    for index in range(num_query):
        mn = np.minimum(v[index], v[num_query:])
        mx = np.maximum(v[index], v[num_query:])
        jaccard[index] = 1 - mn.sum(axis=1) / (mx.sum(axis=1) + 1e-12)
    return jaccard * (1 - lam) + original_dist[:num_query, num_query:] * lam

def full_eval(model, query_loader, gallery_loader, device="cuda", rerank=True):
    qf, qp, qc = extract_features(model, query_loader, device=device)
    gf, gp, gc = extract_features(model, gallery_loader, device=device)
    if qf.shape[0] == 0 or gf.shape[0] == 0:
        print("  [WARN] Empty query or gallery -- skipping eval")
        return 0.0, np.zeros(50), None, None
    dist = 1.0 - qf @ gf.T
    map_score, cmc = eval_market1501(dist, qp, gp, qc, gc)
    map_rr, cmc_rr = None, None
    if rerank:
        dist_rr = compute_reranking(qf, gf)
        map_rr, cmc_rr = eval_market1501(dist_rr, qp, gp, qc, gc)
    return map_score, cmc, map_rr, cmc_rr

print("Evaluation functions ready")

## 5. CLIP RN50x4 ReID Model

In [ ]:
class CLIPResNetReID(nn.Module):
    """CLIP RN50x4 visual encoder + projection + BNNeck + classifier."""

    def __init__(self, num_classes, backbone_dim=BACKBONE_OUT_DIM, embed_dim=EMBED_DIM):
        super().__init__()
        clip_model = open_clip.create_model("RN50x4", pretrained="openai")
        self.backbone = clip_model.visual
        self.backbone_dim = backbone_dim
        self.embed_dim = embed_dim

        self.proj = nn.Linear(self.backbone_dim, self.embed_dim, bias=False)
        nn.init.kaiming_normal_(self.proj.weight, mode="fan_out")

        self.bn = nn.BatchNorm1d(self.embed_dim)
        self.bn.bias.requires_grad_(False)

        self.cls_head = nn.Linear(self.embed_dim, num_classes, bias=False)
        nn.init.normal_(self.cls_head.weight, std=0.001)

        del clip_model

    def forward(self, x):
        backbone_feat = self.backbone(x)
        feat = self.proj(backbone_feat)
        bn_feat = self.bn(feat)
        if self.training:
            logits = self.cls_head(bn_feat)
            return logits, feat
        return F.normalize(bn_feat, p=2, dim=1)

    def get_param_groups(self, backbone_lr, head_lr):
        head_params = list(self.proj.parameters()) + list(self.bn.parameters()) + list(self.cls_head.parameters())
        return [
            {"params": list(self.backbone.parameters()), "lr": backbone_lr},
            {"params": head_params, "lr": head_lr},
        ]

model = CLIPResNetReID(num_classes=num_classes, backbone_dim=BACKBONE_OUT_DIM, embed_dim=EMBED_DIM).to(DEVICE)
if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"Wrapped model in DataParallel ({NUM_GPUS} GPUs)")

raw_model = model.module if hasattr(model, "module") else model
print(f"Model parameters: {sum(param.numel() for param in raw_model.parameters()):,}")
print(f"Backbone output dim: {raw_model.backbone_dim}")
print(f"Embedding dim: {raw_model.embed_dim}")

## 6. Training

Recipe:
- CLIP RN50x4 visual encoder with **288x288** inputs
- **AdamW** with 2 LR groups: backbone `1e-4`, head `1e-3`
- **200 epochs**, **10 epoch warmup**, cosine decay, **EMA=0.9999**
- **CE+LS(0.05)** + **Triplet(m=0.3)** + delayed **CenterLoss(5e-4 @ epoch 15)**
- Eval every **20 epochs** using the EMA model

In [ ]:
ce_loss = CrossEntropyLabelSmooth(num_classes, epsilon=0.05).to(DEVICE)
triplet_loss_fn = TripletLossHardMining(margin=0.3).to(DEVICE)
center_loss_fn = CenterLoss(num_classes, EMBED_DIM).to(DEVICE)

optimizer = torch.optim.AdamW(
    raw_model.get_param_groups(BACKBONE_LR, HEAD_LR),
    weight_decay=WEIGHT_DECAY,
)
center_optimizer = SGD(center_loss_fn.parameters(), lr=0.5)
base_lrs = [group["lr"] for group in optimizer.param_groups]

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
ema = ModelEMA(raw_model, decay=EMA_DECAY)

def set_epoch_lrs(epoch):
    if epoch < WARMUP_EPOCHS:
        factor = (epoch + 1) / WARMUP_EPOCHS
    else:
        progress = epoch - WARMUP_EPOCHS
        cosine_span = max(EPOCHS - WARMUP_EPOCHS, 1)
        factor = 0.5 * (1.0 + math.cos(math.pi * progress / cosine_span))
    for index, group in enumerate(optimizer.param_groups):
        group["lr"] = base_lrs[index] * factor

history = {
    "loss": [],
    "mAP": [],
    "R1": [],
    "mAP_rr": [],
    "R1_rr": [],
    "eval_epoch": [],
}

best_model_path = OUTPUT_DIR / "clip_rn50x4_cityflowv2_best_ema.pth"
last_model_path = OUTPUT_DIR / "clip_rn50x4_cityflowv2_last_ema.pth"
best_mAP = 0.0

print("=" * 72)
print(f"Training CLIP RN50x4 on CityFlowV2 ({num_classes} IDs, {num_cameras} cameras)")
print(f"Backbone LR: {BACKBONE_LR} | Head LR: {HEAD_LR} | Weight decay: {WEIGHT_DECAY}")
print(f"Epochs: {EPOCHS} | Warmup: {WARMUP_EPOCHS} | Batch: {BATCH_SIZE}")
print(f"Losses: CE+LS(0.05) + Triplet(m=0.3) + CenterLoss(weight={CENTER_WEIGHT}, start={CENTER_START_EPOCH})")
print(f"EMA: enabled (decay={EMA_DECAY})")
print("=" * 72)

start_time = time.time()
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    num_batches = 0
    use_center = epoch >= CENTER_START_EPOCH

    set_epoch_lrs(epoch)

    for imgs, pids, cams, _ in train_loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        pids = pids.to(DEVICE).long()
        cams = cams.to(DEVICE).long()

        optimizer.zero_grad(set_to_none=True)
        if use_center:
            center_optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits, feats = model(imgs)
            id_loss = ce_loss(logits, pids)
            metric_loss = triplet_loss_fn(feats, pids)
            total_loss = id_loss + metric_loss
            if use_center:
                center_value = center_loss_fn(feats.float(), pids)
                total_loss = total_loss + CENTER_WEIGHT * center_value

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        if use_center:
            scaler.unscale_(center_optimizer)
        torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        if use_center:
            scaler.step(center_optimizer)
        scaler.update()
        ema.update(raw_model)

        running_loss += float(total_loss.detach().item())
        num_batches += 1

    epoch_loss = running_loss / max(num_batches, 1)
    history["loss"].append(epoch_loss)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        head_lr_now = optimizer.param_groups[-1]["lr"]
        center_tag = " [+center]" if use_center else ""
        print(f"Epoch {epoch + 1:3d} | Loss {epoch_loss:.4f} | head_lr={head_lr_now:.2e}{center_tag}")

    if CAN_EVAL and ((epoch + 1) % EVAL_EVERY == 0 or epoch == EPOCHS - 1):
        map_score, cmc, map_rr, cmc_rr = full_eval(ema.ema, query_loader, gallery_loader, device=DEVICE, rerank=True)
        history["mAP"].append(map_score)
        history["R1"].append(float(cmc[0]))
        history["mAP_rr"].append(float(map_rr) if map_rr is not None else 0.0)
        history["R1_rr"].append(float(cmc_rr[0]) if cmc_rr is not None else 0.0)
        history["eval_epoch"].append(epoch + 1)

        is_best = map_score > best_mAP
        if is_best:
            best_mAP = map_score
            torch.save(ema.ema.state_dict(), best_model_path)

        print(f"Eval @ epoch {epoch + 1:3d}:")
        print(f"  EMA   mAP: {map_score:.4f}, R1: {cmc[0]:.4f}{' BEST' if is_best else ''}")
        if map_rr is not None:
            print(f"  EMA   mAP(RR): {map_rr:.4f}, R1(RR): {cmc_rr[0]:.4f}")

if not best_model_path.exists():
    torch.save(ema.ema.state_dict(), best_model_path)

torch.save(ema.ema.state_dict(), last_model_path)
elapsed = time.time() - start_time
print(f"\nDone in {elapsed / 3600:.1f}h | Best mAP: {best_mAP:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("CLIP RN50x4 -- CityFlowV2 Vehicle ReID", fontsize=14, fontweight="bold")

ax = axes[0]
loss_epochs = list(range(1, len(history["loss"]) + 1))
ax.plot(loss_epochs, history["loss"], "b-", alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training Loss")
ax.grid(True, alpha=0.3)

ax = axes[1]
eval_epochs = history.get("eval_epoch", [])
if history["mAP"]:
    ax.plot(eval_epochs, [score * 100 for score in history["mAP"]], "r-o", label="mAP")
    ax.plot(eval_epochs, [score * 100 for score in history["R1"]], "b-s", label="R1")
    if any(history["mAP_rr"]):
        ax.plot(eval_epochs, [score * 100 for score in history["mAP_rr"]], "r--^", label="mAP(RR)")
        ax.plot(eval_epochs, [score * 100 for score in history["R1_rr"]], "b--d", label="R1(RR)")
ax.set_xlabel("Epoch")
ax.set_ylabel("%")
ax.set_title("Validation Metrics")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Export Model

In [ ]:
best_state_path = best_model_path if best_model_path.exists() else last_model_path
best_state = torch.load(best_state_path, map_location="cpu", weights_only=False)

export_path = EXPORT_DIR / "clip_rn50x4_cityflowv2.pth"
torch.save({"state_dict": best_state}, export_path)
print(f"Model exported to {export_path} ({export_path.stat().st_size / 1e6:.1f} MB)")

metadata = {
    "task": "vehicle_reid",
    "dataset": "cityflowv2",
    "source_dataset": "AI City Challenge 2022 Track 1",
    "model": {
        "architecture": "clip_rn50x4",
        "type": "clip_resnet_reid",
        "clip_pretrained": "openai",
        "backbone_output_dim": BACKBONE_OUT_DIM,
        "embedding_dim": EMBED_DIM,
        "input_size": list(INPUT_SIZE),
        "normalization": {"mean": CLIP_MEAN, "std": CLIP_STD},
        "tricks": [
            "Projection(640->768)",
            "BNNeck",
            "CE+LS(0.05)",
            "TripletLoss(m=0.3)",
            f"CenterLoss(delayed@ep{CENTER_START_EPOCH})",
            "CosLR",
            "RE",
            "CLIP-norm",
            "AugBaseline",
            "EMA",
        ],
        "best_mAP": float(best_mAP),
        "best_mAP_rr": float(max(history["mAP_rr"])) if history["mAP_rr"] else None,
        "epochs": EPOCHS,
        "backbone_lr": BACKBONE_LR,
        "head_lr": HEAD_LR,
        "ema_decay": EMA_DECAY,
        "center_loss_weight": CENTER_WEIGHT,
        "center_loss_start_epoch": CENTER_START_EPOCH,
        "num_classes": num_classes,
        "num_cameras": num_cameras,
    },
    "exports": {
        "best_ema": str(best_state_path),
        "stage2_export": str(export_path),
    },
    "split": {
        "train_images": len(train_data),
        "train_ids": num_classes,
        "query_images": len(query_data),
        "gallery_images": len(gallery_data),
        "eval_ids": len(eval_ids),
    },
    "cameras": CAMERAS,
}

meta_path = EXPORT_DIR / "clip_rn50x4_cityflowv2_metadata.json"
with open(meta_path, "w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2, ensure_ascii=True)
print(f"Metadata saved to {meta_path}")

print("\n" + "=" * 60)
print("RESULTS SUMMARY -- CityFlowV2 Vehicle ReID (CLIP RN50x4)")
print("=" * 60)
print(f"Dataset:    CityFlowV2 ({len(CAMERAS)} cameras)")
print(f"Train:      {len(train_data)} images, {num_classes} IDs")
print(f"Eval:       {len(query_data)} query + {len(gallery_data)} gallery")
print(f"Model:      CLIP RN50x4 visual encoder + 640->768 projection")
print(f"Epochs:     {EPOCHS}")
print(f"EMA:        enabled (decay={EMA_DECAY})")
if history["mAP"]:
    print(f"Best mAP:   {best_mAP * 100:.2f}%")
    print(f"Best R1:    {max(history['R1']) * 100:.2f}%")
    if history["mAP_rr"] and any(history["mAP_rr"]):
        print(f"Best mAP(RR): {max(history['mAP_rr']) * 100:.2f}%")
print("=" * 60)

## 8. Inference Integration

Use the exported checkpoint in stage 2 as an additional vehicle ReID model:

```yaml
stage2:
  reid:
    vehicle3:
      enabled: true
      save_separate: true
      model_name: "clip_rn50x4"
      weights_path: "models/reid/clip_rn50x4_cityflowv2.pth"
      embedding_dim: 768
      input_size: [288, 288]
      clip_normalization: true
```

Expected loader shape:

```python
clip_model = open_clip.create_model("RN50x4", pretrained=None)
backbone = clip_model.visual
model = CLIPResNetReID(num_classes=1, backbone_dim=640, embed_dim=768)
model.backbone = backbone
ckpt = torch.load("models/reid/clip_rn50x4_cityflowv2.pth", map_location="cpu", weights_only=False)
state = ckpt.get("state_dict", ckpt)
model.load_state_dict(state, strict=False)
model.eval()
```